In [ ]:
# %%
import pandas as pd
import numpy as np
import re
import unicodedata
from datetime import date
from currency_converter import CurrencyConverter

c = CurrencyConverter()

# -------------------------
# Config
# -------------------------
IN_PATH = "../Moritz/survey_results_public.csv"
OUT_PATH = "survey_results_cleaned.csv"

drop_cols = [
    'EmploymentAddl', 'LearnCodeChoose', 'LearnCode', 'AILearnHow', 'PurchaseInfluence',
    'ToolCountWork', 'ToolCountPersonal',
    'LanguageAdmired', 'LanguagesHaveEntry', 'LanguagesWantEntry',
    'DatabaseAdmired', 'DatabaseHaveEntry', 'DatabaseWantEntry',
    'PlatformAdmired', 'PlatformHaveEntry', 'PlatformWantEntry',
    'WebframeAdmired', 'WebframeHaveEntry', 'WebframeWantEntry',
    'DevEnvsAdmired', 'DevEnvHaveEntry', 'DevEnvWantEntry',
    'OpSysPersonal use', 'OpSysProfessional use',
    'OfficeStackAsyncAdmired', 'OfficeStackHaveEntry', 'OfficeStackWantEntry',
    'CommPlatformAdmired', 'CommPlatformHaveEntr', 'CommPlatformWantEntr',
    'AIModelsAdmired', 'AIModelsHaveEntry', 'AIModelsWantEntry',
    'AISent', 'AIAcc', 'AIComplex',
    'AIToolCurrently partially AI', "AIToolDon't plan to use AI for this task",
    'AIToolPlan to partially use AI', 'AIToolPlan to mostly use AI',
    'AIToolCurrently mostly AI', 'AIFrustration', 'AIExplain',
    'AIAgentChange', 'AgentUsesGeneral',
    'AIAgentImpactSomewhat agree', 'AIAgentImpactNeutral',
    'AIAgentImpactSomewhat disagree', 'AIAgentImpactStrongly agree',
    'AIAgentImpactStrongly disagree',
    'AIAgentChallengesNeutral', 'AIAgentChallengesSomewhat disagree',
    'AIAgentChallengesStrongly agree', 'AIAgentChallengesSomewhat agree',
    'AIAgentChallengesStrongly disagree',
    'AIAgentKnowledge', 'AIAgentKnowWrite',
    'AIAgentOrchestration', 'AIAgentOrchWrite',
    'AIAgentObserveSecure', 'AIAgentObsWrite',
    'AIAgentExternal', 'AIAgentExtWrite',
    'AIHuman', 'AIOpen'
]
prefixes_drop = ("TechEndorse", "TechOppose", "JobSatPoints", "SO")

remote_map = {
    "Remote": 0,
    "In-person": 1,
    "Hybrid (some remote, leans heavy to in-person)": 0.75,
    "Hybrid (some in-person, leans heavy to flexibility)": 0.25,
    "Your choice (very flexible, you can come in when you want or just as needed)": 0.5
}

age_map = {
    "Under 18 years old": 17,
    "18-24 years old": 21,
    "25-34 years old": 29,
    "35-44 years old": 39,
    "45-54 years old": 49,
    "55-64 years old": 59,
    "65 years or older": 70
}
age_map2 = {
    "Under 18 years old": 18,
    "18-24 years old": 24,
    "25-34 years old": 34,
    "35-44 years old": 44,
    "45-54 years old": 54,
    "55-64 years old": 64,
    "65 years or older": 100
}

multi_select_cols = [
    'LanguageHaveWorkedWith', 'LanguageWantToWorkWith',
    'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith',
    'PlatformHaveWorkedWith', 'PlatformWantToWorkWith',
    'WebframeHaveWorkedWith', 'WebframeWantToWorkWith',
    'DevEnvsHaveWorkedWith', 'DevEnvsWantToWorkWith',
    'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith',
    'AIModelsHaveWorkedWith', 'AIModelsWantToWorkWith',
    'AIAgent_Uses'
]

# -------------------------
# Helpers
# -------------------------
def insert_mapped_column(df, base_col, new_col, mapping):
    if base_col not in df.columns:
        return
    df.insert(df.columns.get_loc(base_col) + 1, new_col, df[base_col].map(mapping))

def clean_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = unicodedata.normalize("NFC", s)
    s = s.replace("–", "-").replace("—", "-").replace("’", "'")
    s = re.sub(r"\s+", " ", s)
    return s

def to_lowercase(s):
    if pd.isna(s):
        return s
    return str(s).lower()

def clean_multi_select_to_str(value):
    """'A;B;C' -> 'a;b;c' (unique+sorted). NaN -> '' """
    if pd.isna(value):
        return ""
    parts = [clean_text(p).lower() for p in str(value).split(";")]
    parts = sorted(set([p for p in parts if p]))
    return ";".join(parts)

def parse_years(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s == "":
        return np.nan
    if "less than 1" in s:
        return 0.5
    if "more than 50" in s:
        return 51.0
    try:
        return float(s)
    except:
        return np.nan

def convert_to_usd(currency, comp):
    if pd.isna(currency) or pd.isna(comp):
        return np.nan
    if currency not in c.currencies:
        return np.nan

    if currency == "RUB":
        return c.convert(comp, currency, "USD", date=date(2022, 3, 1))
    elif currency == "HRK":
        return c.convert(comp, currency, "USD", date=date(2022, 12, 30))
    else:
        return c.convert(comp, currency, "USD", date=date(2025, 10, 6))

# -------------------------
# Cleanup pipeline
# -------------------------
df = pd.read_csv(IN_PATH)

# Drop unneeded columns (robust)
df = df.drop(columns=drop_cols, errors="ignore")
df = df.drop(columns=df.columns[df.columns.str.startswith(prefixes_drop)], errors="ignore")

# EdLevel shorten
if "EdLevel" in df.columns:
    df["EdLevel"] = df["EdLevel"].astype(str).str.split("(").str[0].str.strip()

# Remote mapping + missing flag + fill (0.5)
insert_mapped_column(df, "RemoteWork", "RemoteCategoryNum", remote_map)
if "RemoteCategoryNum" in df.columns:
    df["RemoteMissing"] = df["RemoteCategoryNum"].isna().astype("int8")
    df["RemoteCategoryNum"] = df["RemoteCategoryNum"].fillna(0.5)

# Age mapping + filter <=65
insert_mapped_column(df, "Age", "AgeNum", age_map)
insert_mapped_column(df, "Age", "MaxAge", age_map2)
if "AgeNum" in df.columns:
    df = df[df["AgeNum"].notna() & (df["AgeNum"] <= 65)].copy()

# YearsCode / WorkExp numeric
if "YearsCode" in df.columns:
    df["YearsCode"] = df["YearsCode"].apply(parse_years)
if "WorkExp" in df.columns:
    df["WorkExp"] = pd.to_numeric(df["WorkExp"], errors="coerce")

# Lowercase object columns except Country/Currency
exclude_obj = {"Country", "Currency"}
for col in df.select_dtypes(include=["object"]).columns:
    if col not in exclude_obj:
        df[col] = df[col].apply(to_lowercase)

# Multi-select cleanup (keep as string with ;)
for col in [c for c in multi_select_cols if c in df.columns]:
    df[col] = df[col].apply(clean_multi_select_to_str)

# Plausibility filters
if {"WorkExp", "MaxAge"}.issubset(df.columns):
    df = df[~(df["WorkExp"] > (df["MaxAge"] - 16))].copy()
if {"YearsCode", "MaxAge"}.issubset(df.columns):
    df = df[~(df["YearsCode"] > (df["MaxAge"] - 6))].copy()

# Salary conversion
if "Currency" in df.columns:
    df["Currency"] = df["Currency"].astype(str).str[:3]
    df.loc[df["Currency"].isin(["nan", "none", "None"]), "Currency"] = np.nan
if "CompTotal" in df.columns:
    df["CompTotal"] = pd.to_numeric(df["CompTotal"], errors="coerce")

if {"Currency", "CompTotal"}.issubset(df.columns):
    df["ConvertedCompTotal"] = df.apply(lambda r: convert_to_usd(r["Currency"], r["CompTotal"]), axis=1)

# Drop raw salary cols AFTER conversion
df = df.drop(columns=[c for c in ["CompTotal", "Currency", "ConvertedCompYearly"] if c in df.columns], errors="ignore")

# 95%-Filter (row removal) for numeric outliers
for col in [c for c in ["ConvertedCompTotal", "WorkExp", "YearsCode"] if c in df.columns]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    q95 = df[col].quantile(0.95)
    before = len(df)
    df = df[df[col].isna() | (df[col] <= q95)].copy()
    print(f"{col}: kept <= q95={q95:.4g} | removed {before - len(df)} rows")

# Save
df.to_csv(OUT_PATH, index=False)
print("Saved:", df.shape, "->", OUT_PATH)
